In [4]:
%pip install pandas numpy altair

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
%pip install pandas numpy altair
import pandas as pd
import numpy as np
import altair as alt

url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/building_inventory.csv"

df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Dataset shape: (8862, 22)
Columns: ['Agency Name', 'Location Name', 'Address', 'City', 'Zip code', 'County', 'Congress Dist', 'Congressional Full Name', 'Rep Dist', 'Rep Full Name', 'Senate Dist', 'Senator Full Name', 'Bldg Status', 'Year Acquired', 'Year Constructed', 'Square Footage', 'Total Floors', 'Floors Above Grade', 'Floors Below Grade', 'Usage Description', 'Usage Description 2', 'Usage Description 3']


,Agency Name,Location Name,Address,City,Zip code,County,Congress Dist,Congressional Full Name,Rep Dist,Rep Full Name,...,Bldg Status,Year Acquired,Year Constructed,Square Footage,Total Floors,Floors Above Grade,Floors Below Grade,Usage Description,Usage Description 2,Usage Description 3
0,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,1975,1975,144,1,1,0,Unusual,Unusual,Not provided
1,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
2,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
3,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
4,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided


In [7]:
# Prepare data for Visualization 1
df_area = df[['Agency Name', 'Square Footage']].copy()

df_area['Square Footage'] = pd.to_numeric(
    df_area['Square Footage'], errors='coerce'
)

df_area = df_area.dropna()
df_area = df_area[df_area['Square Footage'] > 0]

agency_area = (
    df_area.groupby('Agency Name')['Square Footage']
    .sum()
    .reset_index()
    .rename(columns={'Square Footage': 'Total Square Footage'})
    .sort_values('Total Square Footage', ascending=False)
    .head(12)
)

# Create bar chart
chart1 = (
    alt.Chart(agency_area)
    .mark_bar()
    .encode(
        x=alt.X(
            'Total Square Footage:Q',
            title='Total Square Footage'
        ),
        y=alt.Y(
            'Agency Name:N',
            sort='-x',
            title='Agency'
        ),
        color=alt.Color(
            'Total Square Footage:Q',
            scale=alt.Scale(scheme='blues')
        ),
        tooltip=[
            'Agency Name:N',
            'Total Square Footage:Q'
        ]
    )
    .properties(
        title='Top 12 Agencies by Total Building Area',
        width=650,
        height=400
    )
)

chart1


alt.Chart(...)

In [8]:
# Prepare data for Visualization 2
df_scatter = df[
    ['Year Constructed', 'Square Footage',
     'Agency Name', 'Bldg Status']
].copy()

df_scatter['Year Constructed'] = pd.to_numeric(
    df_scatter['Year Constructed'], errors='coerce'
)
df_scatter['Square Footage'] = pd.to_numeric(
    df_scatter['Square Footage'], errors='coerce'
)

df_scatter = df_scatter.dropna(
    subset=['Year Constructed', 'Square Footage']
)

df_scatter = df_scatter[
    (df_scatter['Year Constructed'].between(1800, 2026)) &
    (df_scatter['Square Footage'] > 0)
]

df_scatter['Bldg Status'] = df_scatter['Bldg Status'].fillna('Unknown')

# Sample data to keep the interactive chart responsive
df_scatter = df_scatter.sample(
    n=min(1800, len(df_scatter)),
    random_state=42
)

# Create interactive scatter plot
chart2 = (
    alt.Chart(df_scatter)
    .mark_circle(size=55, opacity=0.55)
    .encode(
        x=alt.X(
            'Year Constructed:Q',
            title='Year Constructed',
            scale=alt.Scale(zero=False)
        ),
        y=alt.Y(
            'Square Footage:Q',
            title='Square Footage (log scale)',
            scale=alt.Scale(type='log')
        ),
        color=alt.Color(
            'Bldg Status:N',
            title='Building Status'
        ),
        tooltip=[
            alt.Tooltip('Agency Name:N', title='Agency'),
            alt.Tooltip('Year Constructed:Q', title='Year'),
            alt.Tooltip('Square Footage:Q', title='Square Footage', format=',.0f'),
            alt.Tooltip('Bldg Status:N', title='Status')
        ]
    )
    .properties(
        title='Building Construction Year vs. Square Footage',
        width=650,
        height=400
    )
    .interactive()
)

chart2


alt.Chart(...)

In [9]:
from pathlib import Path

project = Path.cwd()

if not (project / "_config.yml").exists():
    project = project.parent

output_dir = project / "assets" / "json"
output_dir.mkdir(parents=True, exist_ok=True)

chart1.save(str(output_dir / "hw5_chart1.json"))
chart2.save(str(output_dir / "hw5_chart2.json"))

print("Chart 1 saved:", (output_dir / "hw5_chart1.json").exists())
print("Chart 2 saved:", (output_dir / "hw5_chart2.json").exists())


Chart 1 saved: True
Chart 2 saved: True
